In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn opencv-python imageio-ffmpeg

# One sign classifier for all pose models

Does a model's keypoints still allow the signs to be recognized when the video gets worse?

**One** classifier is trained on the clean (`original`) keypoints of the training videos of **all models together**, and then used,
unchanged, on the held-out videos in every degraded scenario of every model. The videos (`eval_setup.txt`), the preprocessing, the
degradations (`tests.txt`, the same names, levels and random seeds) and the classifier are identical, so a difference between the models
comes from the pose estimator alone.

**Before running**
1. `mediapipe_classifier.ipynb` and then `mediapipe_to_keypoints.ipynb` → `keypoints/kp_mediapipe_<scenario>/`
2. `sapiens_pipeline.ipynb` (it runs `sapiens_to_keypoints.ipynb` at its end) → `keypoints/kp_sapiens_<scenario>/`

Both read the same `eval_setup.txt`. Every file has the same format: `kp2d` `[frames, 52, 2]`, `conf`, `joint_names`.

* **Split**: `test_share` of the videos of every sign (at least one) are held out and never used for training. The split is fixed and saved as `classifier_split.csv`.
* **Features**: shoulders, elbows, wrists and the finger joints of both hands, relative to the shoulders, at `frames` evenly spaced moments of the sign. Joints below `min_conf` count as missing.
* **accuracy_retained** = accuracy / the same model's accuracy on its own original videos. A level passes while it stays at or above `limit`.
* With few test videos accuracy moves in coarse steps (10 test videos = steps of 0.1), so more `videos_per_sign` gives smoother curves.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from degradations import load_setup, load_tests, select_videos, make_scenarios
import sign_eval

SETUP_FILE = "eval_setup.txt"            # the same files the pose notebooks read
SETUP      = load_setup(SETUP_FILE)
TESTS      = load_tests("tests.txt")

MODELS   = ["mediapipe", "sapiens"]      # keypoints/kp_<model>_<scenario>/<sign>/<video>.npz
KP_ROOT  = "keypoints"
RUN_NAME = "model_comparison"            # results go to results/<RUN_NAME>/

videos    = select_videos(SETUP["videos"])
scenarios = make_scenarios(TESTS, SETUP["run_tests"], SETUP["random_seeds"])
out = Path("results") / RUN_NAME
out.mkdir(parents=True, exist_ok=True)
print(f"{len({v['sign'] for v in videos})} signs, {len(videos)} videos, {len(scenarios)} scenarios, models: {MODELS}")

## Train the classifier, test every scenario

In [ ]:
predictions, split, skipped = sign_eval.evaluate(videos, scenarios, MODELS, SETUP["classifier"], KP_ROOT)
split.to_csv(out / "classifier_split.csv", index=False)
predictions.to_csv(out / "classifier_predictions.csv", index=False)
if skipped:
    print("left out, keypoints missing for a model:", skipped)

## Accuracy per scenario

In [ ]:
summary, thresholds = sign_eval.summarize(predictions, SETUP["classifier"]["limit"])
summary.to_csv(out / "classifier_summary.csv", index=False)
thresholds.to_csv(out / "classifier_thresholds.csv", index=False)
(summary.pivot_table(index=["level_index", "scenario"], columns="model", values=["accuracy", "accuracy_retained"])
        .reset_index(level=0, drop=True).round(3))

### Where does each model stop recognizing the signs?
`last_ok_level` is the most degraded level that still keeps the `limit` share of the model's original accuracy; `first_bad_level` is the first one below it.

In [ ]:
thresholds

## Accuracy against the degradation level
Solid = accuracy, dashed = that model's limit (`limit` x its original accuracy), dotted grey = chance.

In [ ]:
COLORS = {m: c for m, c in zip(MODELS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"])}
STYLES = ["-", "--", ":", "-."]
MARKER_SIZE = {m: 9 - 3 * n for n, m in enumerate(MODELS)}   # different sizes, so models with identical accuracy both stay visible
limit, chance = SETUP["classifier"]["limit"], 1 / split.sign.nunique()
degraded = summary[summary.test != "original"]
tests = list(dict.fromkeys(degraded.test))
ncols = min(4, len(tests))
nrows = -(-len(tests) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.8 * ncols, 3.3 * nrows), squeeze=False)
for ax in axes.ravel()[len(tests):]:
    ax.axis("off")
for ax, test in zip(axes.ravel(), tests):
    g = degraded[degraded.test == test]
    for n, model in enumerate(MODELS):
        gm = g[g.model == model]
        orig = summary[(summary.model == model) & (summary.test == "original")].accuracy.iloc[0]
        for i, (order, go) in enumerate(gm.groupby("order", dropna=False)):
            go = go.groupby(["level_index", "level"])["accuracy"].agg(["mean", "std"]).reset_index()
            y = np.r_[orig, go["mean"].values]
            err = np.r_[0, go["std"].fillna(0).values]
            x = np.arange(len(y))
            label = model + (f" ({order})" if gm.order.nunique() > 1 else "")
            ax.plot(x, y, "o", ls=STYLES[i % len(STYLES)], color=COLORS[model], lw=2, ms=MARKER_SIZE[model], zorder=3 + n, label=label)
            ax.fill_between(x, y - err, y + err, color=COLORS[model], alpha=0.12, lw=0)
        ax.axhline(orig * limit, color=COLORS[model], ls="--", lw=0.8, alpha=0.5)
    levels = g[g.model == MODELS[0]].drop_duplicates("level_index").sort_values("level_index").level
    ax.set_xticks(np.arange(len(levels) + 1))
    ax.set_xticklabels(["original"] + list(levels), rotation=30, ha="right", fontsize=8)
    ax.axhline(chance, color="#888888", ls=":", lw=1)
    ax.set_ylim(0, 1.05)
    ax.set_title(test, fontsize=10, loc="left")
    ax.set_xlabel("mild → severe", fontsize=8)
    ax.grid(color="#e1e0d9", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=7, frameon=False)
axes[0][0].set_ylabel("accuracy", fontsize=8)
fig.suptitle("One sign classifier on the keypoints of each pose model", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
fig.savefig(out / "classifier_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()